# Title: Feature Engineering
### Author: Kolbe Sussman

In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations
import ast
import os
from tqdm import tqdm
import random

In [2]:
AUTHOR_METRICS = "../data/processed/author_network_metrics.csv"
AUTHOR_EDGES = "../data/processed/author_network_edges.csv"
AFFILIATION_METRICS = "../data/processed/affiliation_network_metrics.csv"
TOPIC_METRICS = "../data/processed/topic_network_metrics.csv"
WORKS_CSV = "../data/processed/umich_works_cleaned.csv"
OUTPUT_CSV = "../data/processed/features_pairs.csv"

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

In [3]:
author_metrics = pd.read_csv(AUTHOR_METRICS).set_index('author')
affil_metrics = pd.read_csv(AFFILIATION_METRICS, engine='python', on_bad_lines='skip').set_index('affiliation')
topic_metrics = pd.read_csv(TOPIC_METRICS).set_index('topic')

# Build author graph from edges
edges_df = pd.read_csv(AUTHOR_EDGES)
G_author = nx.from_pandas_edgelist(edges_df, 'author_1', 'author_2', edge_attr='weight')

# Load works for metadata
df = pd.read_csv(WORKS_CSV)

In [4]:
list_cols = ['author_names', 'raw_affiliations', 'display_names']
for col in list_cols:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [5]:
df_exploded = df.explode('author_names')

author_papers = df_exploded.groupby('author_names')['id'].count().to_dict()
author_citations = df_exploded.groupby('author_names')['cited_by_count'].sum().to_dict()
author_topics = df_exploded.groupby('author_names')['display_names'].sum().apply(set).to_dict()
author_depts = df_exploded.groupby('author_names')['raw_affiliations'].sum().apply(set).to_dict()


In [6]:
positive_pairs = list(G_author.edges)

all_authors = list(G_author.nodes)
negative_pairs = set()
while len(negative_pairs) < len(positive_pairs):
    a1, a2 = random.sample(all_authors, 2)
    if not G_author.has_edge(a1, a2):
        negative_pairs.add((a1, a2))

pairs = [(a1, a2, 1) for a1, a2 in positive_pairs] + [(a1, a2, 0) for a1, a2 in negative_pairs]


In [7]:
def jaccard(set1, set2):
    if not set1 or not set2:
        return 0
    return len(set1 & set2) / len(set1 | set2)

In [9]:
features = []
for a1, a2, label in tqdm(pairs, desc="Computing features"):
    neighbors1, neighbors2 = set(G_author.neighbors(a1)), set(G_author.neighbors(a2))
    features.append({
        'author_1': a1,
        'author_2': a2,
        'common_neighbors': len(neighbors1 & neighbors2),
        'jaccard': jaccard(neighbors1, neighbors2),
        'degree_diff': abs(author_metrics.loc[a1, 'degree_centrality'] - author_metrics.loc[a2, 'degree_centrality']),
        'eigen_diff': abs(author_metrics.loc[a1, 'eigenvector_centrality'] - author_metrics.loc[a2, 'eigenvector_centrality']),
        'topic_overlap': len(author_topics.get(a1,set()) & author_topics.get(a2,set())),
        'dept_overlap': len(author_depts.get(a1,set()) & author_depts.get(a2,set())),
        'paper_diff': abs(author_papers.get(a1,0) - author_papers.get(a2,0)),
        'citation_diff': abs(author_citations.get(a1,0) - author_citations.get(a2,0)),
        'label': label
    })

Computing features: 100%|██████████| 2997322/2997322 [01:18<00:00, 38008.99it/s]


In [10]:
features_df = pd.DataFrame(features)

In [11]:
features_df.head()

,author_1,author_2,common_neighbors,jaccard,degree_diff,eigen_diff,topic_overlap,dept_overlap,paper_diff,citation_diff,label
0,Alex M. Aisen,Charles R. Meyer,7,0.041420,0.000445,0.000759,6,6,36,8655,1
1,Alex M. Aisen,Leslie E. Quint,15,0.107143,0.000344,0.000693,6,11,25,6669,1
2,Alex M. Aisen,Richard D. Cieslak,4,0.093023,0.000179,0.000010,3,3,7,1327,1
3,Alex M. Aisen,Richard L. Wahl,6,0.037500,0.000397,0.000054,4,8,33,9927,1
4,Alex M. Aisen,Robert A. Koeppe,4,0.009029,0.001758,0.000067,8,9,179,40439,1


In [12]:
features_df.describe()

,common_neighbors,jaccard,degree_diff,eigen_diff,topic_overlap,dept_overlap,paper_diff,citation_diff,label
count,2.997322e+06,2.997322e+06,2.997322e+06,2.997322e+06,2.997322e+06,2.997322e+06,2.997322e+06,2.997322e+06,2997322.0
mean,5.893563e+00,1.803845e-01,1.782297e-04,2.482433e-04,2.143315e+00,7.270920e+00,1.082755e+01,2.319654e+03,0.5
std,7.970706e+00,2.755081e-01,4.098302e-04,7.528502e-03,3.079718e+00,1.858101e+01,2.778612e+01,7.184213e+03,0.5
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0
25%,0.000000e+00,0.000000e+00,1.452503e-05,1.983478e-08,0.000000e+00,0.000000e+00,0.000000e+00,7.100000e+01,0.0
50%,0.000000e+00,0.000000e+00,4.841677e-05,3.548748e-07,1.000000e+00,2.000000e+00,2.000000e+00,3.190000e+02,0.5
75%,1.100000e+01,2.702703e-01,1.355670e-04,3.958934e-06,3.000000e+00,9.000000e+00,7.000000e+00,1.455000e+03,1.0
max,7.990000e+02,1.000000e+00,5.921371e-03,4.061315e-01,1.830000e+02,1.546000e+03,6.380000e+02,1.552620e+05,1.0


In [ ]:

#features_df.to_csv(OUTPUT_CSV, index=False)
